In [1]:
import os
import shutil
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report  # Added for classification report

# Data preparation
train_dir = "/kaggle/input/rice-leaf-diseases-detection/Rice_Leaf_Diease/Rice_Leaf_Diease/train"
test_dir = "/kaggle/input/rice-leaf-diseases-detection/Rice_Leaf_Diease/Rice_Leaf_Diease/test"
working_dir = "/kaggle/working"

# Create working directory structure
os.makedirs(os.path.join(working_dir, "models"), exist_ok=True)
os.makedirs(os.path.join(working_dir, "metrics"), exist_ok=True)
val_dir = os.path.join(working_dir, "val_data")
os.makedirs(val_dir, exist_ok=True)

# Copy and rename validation data
shutil.copytree(test_dir, val_dir, dirs_exist_ok=True)

rename_map = {
    'Neck_Blast': 'neck_blast',
    'Sheath Blight': 'sheath_blight',
    'Rice Hispa': 'rice_hipsa',
    'Tungro': 'tungro'
}

for old, new in rename_map.items():
    old_path = os.path.join(val_dir, old)
    new_path = os.path.join(val_dir, new)
    if os.path.exists(old_path):
        os.rename(old_path, new_path)

# Calculate class weights
class_names = sorted(os.listdir(train_dir))
labels = []
for idx, class_name in enumerate(class_names):
    class_path = os.path.join(train_dir, class_name)
    num_images = len(os.listdir(class_path))
    labels.extend([idx] * num_images)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=np.array(labels)
)
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

# Data generators
img_size = (224, 224)
batch_size = 64

train_datagen = ImageDataGenerator(
    rescale=1/255.,
    horizontal_flip=True,
    zoom_range=0.2,
    rotation_range=10
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False  # Changed to False for consistent evaluation
)

# Model setup
base_model = tf.keras.applications.Xception(
    input_shape=(*img_size, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(*img_size, 3))
x = base_model(inputs)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy", "precision", "recall"]
)

# Callbacks
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001)
]

# Training
history = model.fit(
    train_gen,
    validation_data=val_gen,
    class_weight=class_weights,
    epochs=30,
    verbose=1,
    callbacks=callbacks
)

# Save everything
model.save(os.path.join(working_dir, "models/leaf_disease_xception.keras"))
model.save(os.path.join(working_dir, "models/leaf_disease_xception.h5"))

pd.DataFrame(history.history).to_csv(
    os.path.join(working_dir, "metrics/training_history.csv"), 
    index=False
)

with open(os.path.join(working_dir, "models/class_indices.json"), "w") as f:
    json.dump({v: k for k, v in train_gen.class_indices.items()}, f)

# Generate and print classification report
print("\nGenerating classification report...")
val_gen.reset()  # Reset generator before prediction
y_pred = model.predict(val_gen)
y_pred = np.argmax(y_pred, axis=1)
y_true = val_gen.classes

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=list(val_gen.class_indices.keys())
))

print("\nModel and training artifacts saved successfully in /kaggle/working")

2025-08-02 17:00:11.994293: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754154012.198842      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754154012.261199      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Class Weights: {0: 1.083910533910534, 1: 1.0150675675675676, 2: 1.0075788061703554, 3: 0.8341476957245975, 4: 0.8995808383233533, 5: 1.0609463276836157, 6: 1.5023, 7: 1.028268309377139, 8: 0.9520278833967047, 9: 0.8633908045977011}
Found 15023 images belonging to 10 classes.
Found 3422 images belonging to 10 classes.


I0000 00:00:1754154064.526889      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30


I0000 00:00:1754154080.125068      71 service.cc:148] XLA service 0x7de41843cb50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1754154080.125762      71 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1754154081.276867      71 cuda_dnn.cc:529] Loaded cuDNN version 90300


  1/235 ━━━━━━━━━━━━━━━━━━━━ 1:08:58 18s/step - accuracy: 0.0625 - loss: 3.0618 - precision: 0.0000e+00 - recall: 0.0000e+00

I0000 00:00:1754154088.637757      71 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


235/235 ━━━━━━━━━━━━━━━━━━━━ 536s 2s/step - accuracy: 0.2753 - loss: 2.2993 - precision: 0.3961 - recall: 0.1474 - val_accuracy: 0.6560 - val_loss: 1.1902 - val_precision: 0.9082 - val_recall: 0.2861 - learning_rate: 1.0000e-04
Epoch 2/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 399s 2s/step - accuracy: 0.6048 - loss: 1.1595 - precision: 0.7335 - recall: 0.4778 - val_accuracy: 0.7487 - val_loss: 0.7890 - val_precision: 0.8608 - val_recall: 0.6146 - learning_rate: 1.0000e-04
Epoch 3/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 407s 2s/step - accuracy: 0.6833 - loss: 0.9106 - precision: 0.7892 - recall: 0.5826 - val_accuracy: 0.7843 - val_loss: 0.6484 - val_precision: 0.8548 - val_recall: 0.7034 - learning_rate: 1.0000e-04
Epoch 4/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 409s 2s/step - accuracy: 0.7277 - loss: 0.7868 - precision: 0.8144 - recall: 0.6428 - val_accuracy: 0.8118 - val_loss: 0.5692 - val_precision: 0.8711 - val_recall: 0.7425 - learning_rate: 1.0000e-04
Epoch 5/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 391s 2s/step - 